# 9.1 Interactive Protocol Schema Dashboard

## Motivation
Mathematical protocol extraction generates dense, multi-layered data. A Security Operations Center (SOC) analyst or reverse engineer cannot effectively parse raw Shannon Entropy arrays, Kullback-Leibler divergences, and categorical Typology vectors simultaneously in a static format. To make this pipeline operationally viable, the theoretical outputs must be fused into an Interactive Visual Analytics Dashboard.

## Research Objective
The objective of this final stage is to build an interactive, browser-based dashboard (via Streamlit and Plotly) that synthesizes all preceding data artifacts. This dashboard will allow users to hover over any byte offset in the protocol, instantly viewing its informational variance, its normalized Field Variability Index (FVI), and its semantic macro-field boundary limits.

## Connection to the Literature
Visual explainability is a core tenet of modern Machine Learning for Cybersecurity. While the *Byte-Pattern-Based* segmentation theory provides the mathematical extraction, frameworks like *Netzob* and *UI-RE* emphasize the necessity of interactive graphical user interfaces (GUIs). An interactive schema viewer bridges the gap between the unsupervised learning engine and the human-in-the-loop analyst, facilitating rapid protocol comprehension.

## Advantages & Limitations
* **Advantages:** Unifies $H(M)$, $FVI(j)$, and $F_i$ into a single composite plane. Enables zooming, panning, and precise tooltip inspections of individual byte offsets, which is critical for analyzing massive 1,500-byte MTU industrial payloads.
* **Limitations:** Browser-based rendering of massive Plotly objects can experience memory latency if the packet payload dimension ($M$) exceeds tens of thousands of bytes (e.g., Jumbo Frames), necessitating down-sampling or chunked rendering in future iterations.


## Architectural Translation: The Composite Visual Plane

To render the interactive dashboard, we must map our disparate 1D mathematical arrays onto a shared 2D Cartesian coordinate system:

1. **The X-Axis (Spatial Dimension):** Represents the byte offset index $j \in [0, M-1]$.
2. **The Primary Y-Axis (Continuous Metrics):** Maps the Shannon Entropy $H(M_j) \in [0, 8]$ and the normalized Field Variability Index $FVI(j) \in [0, 1]$.
3. **The Z-Plane (Background Spans):** The reconstructed fields $F_i$ from our generated schema are projected as colored background spans (rectangles) bounded by $x_{min} = b_i$ and $x_{max} = b_{i+1}$.

The color encoding function $C(F_i)$ maps the semantic typology to a human-readable visual cue:
$$
C(F_i) = 
\begin{cases} 
\text{Green}, & \text{if } \mu_{FVI}(F_i) \le 0.05 \text{ (Static)} \\
\text{Orange}, & \text{if } 0.05 < \mu_{FVI}(F_i) \le 0.30 \text{ (Status)} \\
\text{Red}, & \text{if } \mu_{FVI}(F_i) > 0.30 \text{ (Dynamic)} 
\end{cases}
$$

## Algorithm Design: Streamlit & Plotly Integration

**Inputs (Deserialized from Disk):**
* `entropy_profile.npy`: The raw bit-variance profile.
* `macro_boundaries.npy`: The finalized boundary offsets.
* `final_protocol_schema.csv`: The compiled semantic blueprint.

**Outputs:**
* `protocol_dashboard.py`: A dynamically generated Python script that launches a Streamlit web application.

**Design Decisions:**
Rather than plotting static matplotlib charts within the Jupyter cell, we will use Jupyter's `%%writefile` magic command to author a standalone Streamlit application. This proves the pipeline's artifacts can be decoupled from the research environment and deployed as a standalone software product.

In [1]:
%%writefile protocol_dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

# --- 1. Page Configuration ---
st.set_page_config(page_title="Protocol Reverse Engineering Dashboard", layout="wide")
st.title("🛡️ Unsupervised Protocol Schema Extractor")
st.markdown("Interactive visualization of byte-pattern-based field segmentation.")

# --- 2. Data Loading ---
@st.cache_data
def load_research_artifacts():
    processed_dir = os.path.join("..", "data", "processed")
    
    entropy = np.load(os.path.join(processed_dir, "entropy_profile.npy"))
    fvi = np.clip(entropy / 8.0, 0, 1)
    schema_df = pd.read_csv(os.path.join(processed_dir, "final_protocol_schema.csv"))
    
    return entropy, fvi, schema_df

try:
    entropy_profile, fvi_profile, protocol_schema = load_research_artifacts()
except FileNotFoundError:
    st.error("Missing data artifacts. Please ensure Pillar 1 notebooks have been executed.")
    st.stop()

M_dimension = len(entropy_profile)

# --- 3. Interactive Plotly Rendering ---
st.subheader("Field Layout & Positional Variance Map")

fig = go.Figure()

# Add Shannon Entropy Line
fig.add_trace(go.Scatter(
    x=list(range(M_dimension)), 
    y=entropy_profile,
    mode='lines+markers',
    name='Shannon Entropy (Bits)',
    line=dict(color='blue', width=2),
    hovertemplate='Offset: %{x}<br>Entropy: %{y:.3f} bits<extra></extra>'
))

# Add FVI Line
fig.add_trace(go.Scatter(
    x=list(range(M_dimension)), 
    y=fvi_profile * 8.0, # Scaled to match the entropy axis for dual-viewing
    mode='lines',
    name='Scaled FVI',
    line=dict(color='purple', width=2, dash='dot'),
    hoverinfo='skip'
))

# Add Semantic Field Background Spans
color_map = {
    "Static Header / Padding": "rgba(44, 160, 44, 0.2)", # Green
    "Status / Opcode": "rgba(255, 127, 14, 0.2)",         # Orange
    "Dynamic Address / Payload": "rgba(214, 39, 40, 0.2)" # Red
}

for index, row in protocol_schema.iterrows():
    # Parse the string offset range '[start : end]' safely
    range_str = row["Offset Range"].strip("[]")
    start_idx, end_idx = map(int, range_str.split(":"))
    end_idx += 1 # Adjust for inclusive visual boundary
    
    typology = row["Semantic Typology"]
    bg_color = color_map.get(typology, "rgba(128, 128, 128, 0.2)")
    
    # Draw background rectangle for the field
    fig.add_vrect(
        x0=start_idx, x1=end_idx,
        fillcolor=bg_color, opacity=1,
        layer="below", line_width=1, line_color="black",
        annotation_text=row["Field ID"] if (end_idx - start_idx) > 1 else "",
        annotation_position="top left",
        annotation_font_size=10, annotation_font_color="black"
    )

fig.update_layout(
    xaxis_title="Byte Position Offset Index",
    yaxis_title="Information Variance Magnitude",
    hovermode="x unified",
    height=500,
    margin=dict(l=0, r=0, t=30, b=0)
)

st.plotly_chart(fig, use_container_width=True)

# --- 4. Tabular Schema Display ---
st.subheader("Reconstructed Protocol Specification")
st.dataframe(protocol_schema, use_container_width=True)

Writing protocol_dashboard.py
